# TRAINING RESNET WITH DATA FROM ONLY release_in_the_wild

In [11]:
import sys
from pathlib import Path

# add parent directory to path to fix imports
parent_dir = Path().resolve().parent
if str(parent_dir) not in sys.path:
    sys.path.insert(0, str(parent_dir))

from embedding_pipeline.stft import stft

import numpy as np
import matplotlib.pyplot as plt
import torch
import torch.nn as nn
import torch.nn.functional as F
import torch.optim as optim
import os
from torch.utils.data import DataLoader, TensorDataset
import random

In [12]:
def process_clip(file_path, max_time_windows=256):
    magnitude_db, _ = stft(file_path)
    stft_tensor = torch.tensor(magnitude_db).unsqueeze(0).float()  # [3, 512, W]
    width = stft_tensor.shape[-1]

    # clip long samples and pad short samples so all tensors have the same width
    if width > max_time_windows:
        stft_tensor = stft_tensor[:, :, :max_time_windows]
    elif width < max_time_windows:
        pad_width = max_time_windows - width
        stft_tensor = torch.nn.functional.pad(stft_tensor, (0, pad_width))

    return stft_tensor

In [13]:
from resnet import ResNet

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

model = ResNet().to(device)
model.eval()

model.load_state_dict(torch.load('resnet_general_checkpoint.pth', map_location=device)['model'])

<All keys matched successfully>

In [14]:
max_time_windows = 256 # max time windows for stft

In [15]:
# fake
model(process_clip('../data/release_in_the_wild/train/fake/5456.wav', max_time_windows).unsqueeze(0).to(device))

tensor([[17.0174]], device='cuda:0', grad_fn=<AddmmBackward0>)

In [16]:
# real
model(process_clip('../data/release_in_the_wild/train/real/2026.wav', max_time_windows).unsqueeze(0).to(device))

tensor([[-4.1012]], device='cuda:0', grad_fn=<AddmmBackward0>)